# Stylizing ViT - Training Demo

This notebook demonstrates how to train (finetune) the **Stylizing ViT** model on a small custom dataset.

For demonstration purposes, we will download a small subset of the [Fitzpatrick17k](https://github.com/mattgroh/fitzpatrick17k) dataset and run a few steps of optimization.

In [13]:
import os
import torch
import requests
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim
from tqdm import tqdm

# Import StylizingViT
from stylizing_vit import create_model

In [14]:
import random
import numpy as np

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed set to {SEED}")

Random seed set to 42


## 1. Create a Demo Dataset

We define a Dataset class that downloads a few images from the Fitzpatrick17k CSV on initialization.

In [15]:
class DemoDataset(Dataset):
    def __init__(self, num_samples=20, cache_dir="demo_data"):
        self.images = []
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
        
        if not os.path.exists(cache_dir):
            os.makedirs(cache_dir)
            
        # Load Metadata
        csv_url = "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/refs/heads/main/fitzpatrick17k.csv"
        df = pd.read_csv(csv_url)
        df = df[df['url'].notna()]
        
        # Sample random rows
        samples = df.sample(num_samples, random_state=42)
        
        print(f"Downloading {num_samples} images to {cache_dir}...")
        for idx, row in tqdm(samples.iterrows(), total=num_samples):
            img_path = os.path.join(cache_dir, f"{row['md5hash']}.jpg")
            
            if not os.path.exists(img_path):
                try:
                    resp = requests.get(row['url'], timeout=10)
                    if resp.status_code == 200:
                        with open(img_path, 'wb') as f:
                            f.write(resp.content)
                except Exception:
                    continue
            
            # Verify it opens
            try:
                Image.open(img_path).convert('RGB')
                self.images.append(img_path)
            except:
                pass
                
        print(f"Successfully loaded {len(self.images)} images.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path = self.images[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img)

## 2. Setup Training Components

We initialize the model in training mode. This enables the VGG perceptual loss computation internally.

In [16]:
# Configuration
BATCH_SIZE = 4
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataset & Loader
dataset = DemoDataset(num_samples=20)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Model - Note: train=True is crucial
model = create_model(backbone="base", weights="fitzpatrick17k_12_34_56", train=True)
model = model.to(DEVICE)

# Optimization
optimizer = optim.AdamW(model.parameters(), lr=LR)

100%|██████████| 20/20 [00:01<00:00, 16.73it/s]


Successfully loaded 19 images.


## 3. Training Loop

We run a short training loop. The model's forward pass computes the losses automatically when in training mode.
It implicitly uses the batch as both content and style (by rolling the batch for style pairing).

In [17]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    for images in pbar:
        images = images.to(DEVICE)
        
        optimizer.zero_grad()
        
        # The model returns a Loss dataclass and the reconstructed images
        loss_obj, x_recon = model(images)
        
        # Combine losses (using default weights similar to the paper)
        loss_val = (
            70.0 * loss_obj.identity +
            1.0 * loss_obj.consistency +
            10.0 * loss_obj.anatomical +
            7.0 * loss_obj.style
        )
        
        loss_val.backward()
        optimizer.step()
        
        total_loss += loss_val.item()
        pbar.set_postfix({'loss': f"{loss_val.item():.4f}"})
        
    print(f"Epoch {epoch+1} finished with Average Loss: {total_loss / len(dataloader):.4f}")

Epoch 1/10: 100%|██████████| 5/5 [00:00<00:00,  7.41it/s, loss=80.2302]


Epoch 1 finished with Average Loss: 84.3968


Epoch 2/10: 100%|██████████| 5/5 [00:00<00:00,  7.47it/s, loss=83.3980]


Epoch 2 finished with Average Loss: 83.1955


Epoch 3/10: 100%|██████████| 5/5 [00:00<00:00,  7.51it/s, loss=92.2386]


...

Epoch 8 finished with Average Loss: 77.3927


Epoch 9/10: 100%|██████████| 5/5 [00:00<00:00,  7.40it/s, loss=75.3508]


Epoch 9 finished with Average Loss: 76.4890


Epoch 10/10: 100%|██████████| 5/5 [00:00<00:00,  7.39it/s, loss=73.4043]

Epoch 10 finished with Average Loss: 76.0449


## 4. Save the Finetuned Model

You can save the model state dict as usual.

In [18]:
# save_path = "finetuned_stylizing_vit.pth"
# torch.save(model.state_dict(), save_path)
# print(f"Model saved to {save_path}")